# 6.4 - ViT & CLIP: When Transformers Learned to See

**Module 6 - Image & Multimodal Generation** | Netsetos GenAI Engineering

This notebook builds the vision half of multimodal AI from the ground up: you cut an image into patch tokens and run the Module-4 transformer over them (a Vision Transformer), then use a pretrained CLIP model to place images and text in one shared space and measure their match with cosine similarity. From there you reproduce CLIP's contrastive training objective in a few lines, run zero-shot classification with no labels, and switch to the 2026 successor SigLIP 2 to see the softmax-to-sigmoid shift.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Minimal environment prep. Nothing to run yet - this cell just holds the (commented-out) install line and a note that the demos below use seeded/pretrained values, so you don't need to change anything before running.

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install openai torch -q

# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A placeholder setup cell, not a computation. It documents the one dependency install you'd uncomment in a fresh Colab and flags that later cells rely on pretrained weights rather than anything trained here.

**How the code works - step by step**
- **Commented `pip install openai torch -q`** - uncomment only if `torch` (and the OpenAI client, unused here) aren't already present.
- **Comment about reproducibility** - a reminder that the notebook leans on pretrained models and fixed sample inputs, so results are stable across runs.

**In one line:** environment note, no code executes.

**What you'll see:** (no output - environment prep)

## 1 - Fetch a sample image

Every CLIP/SigLIP demo below needs one real picture to embed. This cell downloads a stock photo of a dog once and caches it to disk so the later cells can just `Image.open("dog.jpg")`.

In [ ]:
# Download a sample image used by the CLIP / SigLIP demos below.
import urllib.request, os
if not os.path.exists("dog.jpg"):
    urllib.request.urlretrieve(
        "https://github.com/pytorch/hub/raw/master/images/dog.jpg", "dog.jpg")
print("dog.jpg ready")

**Explanation**

A tiny data-fetch cell, not a model call. It grabs one JPEG from PyTorch's public image hub and saves it locally, guarded so re-running never re-downloads.

**How the code works - step by step**
- **`if not os.path.exists("dog.jpg")`** - skip the download if the file is already on disk (safe to re-run).
- **`urllib.request.urlretrieve(...)`** - pull the sample `dog.jpg` from the pytorch/hub GitHub repo and write it to the working directory.
- **`print("dog.jpg ready")`** - confirm the file is available for the CLIP cells.

**In one line:** download one dog photo, cache it, confirm.

**What you'll see:** Prints `dog.jpg ready`, and a `dog.jpg` file appears in the working directory.

## 2 - Turn an image into patch tokens (a Vision Transformer)

A ViT has exactly one idea: cut the image into fixed patches, treat each patch as a token, and run the Module-4 transformer over the sequence. This cell does that end to end with random weights - patchify to 196 tokens, prepend a `[CLS]` token, add positions, run one encoder layer - to show the image collapse into a single 768-dim embedding.

In [ ]:
import torch, torch.nn as nn

image = torch.randn(1, 3, 224, 224)          # one RGB image (batch of 1)

# 1) cut into 16x16 patches with a strided conv - each patch -> a 768-dim vector
patchify = nn.Conv2d(3, 768, kernel_size=16, stride=16)
tokens = patchify(image).flatten(2).transpose(1, 2)  # (1, 196, 768)
print("patch tokens:", tuple(tokens.shape))     # (224/16)^2 = 196 tokens
# Output: patch tokens: (1, 196, 768)

# 2) prepend a [CLS] token, add position embeddings, run a transformer layer
cls = nn.Parameter(torch.randn(1, 1, 768))       # nn.Parameter = a tensor the optimizer trains (starts random, learned)
pos = nn.Parameter(torch.randn(1, 197, 768))     # one position vector per token
seq = torch.cat([cls, tokens], dim=1) + pos       # (1, 197, 768)

encoder = nn.TransformerEncoderLayer(768, nhead=12, batch_first=True)  # nhead=12: 12 parallel attention heads (Module 4)
image_embedding = encoder(seq)[:, 0]              # the [CLS] row = one image vector
print("image embedding:", tuple(image_embedding.shape))
# Output: image embedding: (1, 768)

**Explanation**

A from-scratch structural demo, not a trained model: it wires up the ViT pipeline with random tensors so you can watch the shapes flow rather than get a meaningful prediction. The takeaway is that a ViT is just the transformer you already know, run over image patches instead of word tokens.

**How the code works - step by step**
- **`patchify = nn.Conv2d(3, 768, kernel_size=16, stride=16)`** - a strided conv with `kernel = stride = 16` slides a non-overlapping 16x16 window, so each patch becomes one 768-dim vector; `.flatten(2).transpose(1,2)` reshapes the 14x14 grid into `(1, 196, 768)` - 196 patch tokens.
- **`cls = nn.Parameter(...)`** - a single learned `[CLS]` token prepended to the sequence; after the transformer its output row stands in for the whole image.
- **`pos = nn.Parameter(...)`** - one learned position vector per token (197 total), added so the model recovers where each patch sat.
- **`nn.TransformerEncoderLayer(768, nhead=12)`** - the Module-4 self-attention layer (12 heads) lets every patch attend to every other.
- **`encoder(seq)[:, 0]`** - take the `[CLS]` output row as the one image embedding.

**In one line:** patchify -> add [CLS] + positions -> transformer -> pool the [CLS] row into one vector.

**What you'll see:** Prints `patch tokens: (1, 196, 768)` then `image embedding: (1, 768)` - 196 patches in, one 768-dim image vector out.

## 3 - CLIP: score an image against captions

Now the real thing. Load a pretrained CLIP model and its processor, then embed one image and three candidate captions into the same shared space and read off which caption matches. This is the whole CLIP interface in a dozen lines - and it runs on CPU.

> **Downloads pretrained weights** from Hugging Face on first run (no API key needed).

In [ ]:
# pip install transformers pillow torch  (runs on CPU; no GPU needed)
from transformers import CLIPModel, CLIPProcessor
from PIL import Image
import torch

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
proc  = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

image = Image.open("dog.jpg")
texts = ["a photo of a dog", "a photo of a cat", "a photo of a car"]

# the processor resizes/normalizes the image and tokenizes the text
inputs = proc(text=texts, images=image, return_tensors="pt", padding=True)
with torch.no_grad():
    out = model(**inputs)

# logits_per_image = cosine similarity (scaled); softmax -> a prob over the 3 texts
probs = out.logits_per_image.softmax(dim=1)[0]
for t, p in zip(texts, probs):
    print(f"{p:.2f}  {t}")
# Output: 0.97  a photo of a dog
# Output: 0.02  a photo of a cat
# Output: 0.01  a photo of a car

**Explanation**

The first live model call: `CLIPModel` bundles both encoders and one forward pass compares the image to every caption. The core idea is that the returned `logits_per_image` are scaled cosine similarities, and softmax turns them into a probability over the captions you supplied.

**How the code works - step by step**
- **`CLIPModel.from_pretrained(...)` / `CLIPProcessor.from_pretrained(...)`** - load the image+text encoders and the preprocessor that resizes/normalizes the image and tokenizes the text.
- **`proc(text=texts, images=image, ...)`** - package the image and the three captions into model-ready tensors in one call.
- **`model(**inputs)`** under `torch.no_grad()` - embed everything into the shared space and return `logits_per_image`, the raw match scores (cosine similarity times CLIP's learned `logit_scale` temperature of ~100).
- **`.softmax(dim=1)[0]`** - normalize across the three captions so they compete and sum to 1; this is a ranking over your captions, not a calibrated 'how doggy' score.

**In one line:** embed image + captions in one space, read scaled cosine similarity as a probability over the captions.

**What you'll see:** Prints the three captions with probabilities, roughly `0.97 a photo of a dog`, `0.02 a photo of a cat`, `0.01 a photo of a car` - CLIP picks 'dog' with high confidence.

## 4 - Contrastive learning: how CLIP trains

CLIP's training is a matching game: score every image against every caption in a batch, then pull the correct pairs together and push the rest apart. This cell reproduces that objective on a toy batch of 4 random image and caption embeddings so you can see the loss is just symmetric cross-entropy on an image-vs-caption grid.

In [ ]:
import torch, torch.nn.functional as F

# A toy batch: 4 image embeddings and their 4 matched caption embeddings.
# (normalize so a dot product IS the cosine similarity)
img_emb = F.normalize(torch.randn(4, 512), dim=1)   # 4 images
txt_emb = F.normalize(torch.randn(4, 512), dim=1)   # their 4 captions

logits = img_emb @ txt_emb.T                        # (4, 4) image-vs-caption grid

# the CORRECT pairs are the diagonal: image i matches caption i
labels = torch.arange(4)                            # [0, 1, 2, 3]
# symmetric cross-entropy: each row picks its caption, each column picks its image
loss = (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2
print("contrastive loss:", round(loss.item(), 2))
# Output: contrastive loss: 1.39   (illustrative - with random init it starts near
# Output: log(4)=1.39 or above and falls toward 0 as the diagonal wins; varies by seed)

**Explanation**

A loss-function demo on random vectors, not real training - the numbers are illustrative, the point is the shape of the objective. The correct pairs are the diagonal of the NxN similarity grid, and the shared space from Step 3 is a by-product of winning this game.

**How the code works - step by step**
- **`F.normalize(torch.randn(4, 512), dim=1)`** - make 4 image and 4 caption embeddings unit-length so a dot product equals cosine similarity.
- **`img_emb @ txt_emb.T`** - the 4x4 grid where entry `(i, j)` is image i's similarity to caption j; the diagonal is the correct pairs.
- **`labels = torch.arange(4)`** - 'row i should pick column i'.
- **`(F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2`** - symmetric cross-entropy: each image picks its caption (rows) and each caption picks its image (columns); pull the diagonal up, push everything else down.

**In one line:** score every image against every caption -> push each toward its diagonal partner via symmetric cross-entropy.

**What you'll see:** Prints something like `contrastive loss: 1.39` - with random init it starts near log(4)=1.39 and would fall toward 0 as the diagonal wins; the exact value varies by seed.

## 5 - Zero-shot classification: no labels needed

Turn CLIP's shared space into a classifier for any classes you can name. Write one text prompt per class, embed the image against all of them, and take the nearest - no training loop, no labeled data. Swap the class list and you have a brand-new classifier instantly.

> Reuses `model`, `proc`, and `image` from Step 3.

In [ ]:
# reuse `model`, `proc`, and `image` from Step 2
classes = ["dog", "cat", "car", "pizza"]
prompts = [f"a photo of a {c}" for c in classes]   # prompt template matters

inputs = proc(text=prompts, images=image, return_tensors="pt", padding=True)
with torch.no_grad():
    probs = model(**inputs).logits_per_image.softmax(dim=1)[0]

best = probs.argmax().item()
print(f"predicted: {classes[best]}  ({probs[best]:.2f})")
# Output: predicted: dog  (0.96)

**Explanation**

A reuse-the-model cell that reframes Step 3 as classification: instead of arbitrary captions, the texts are class names wrapped in a prompt template, and argmax over the similarities is the prediction. 'Zero-shot' means no labeled training set - not that the score is calibrated.

**How the code works - step by step**
- **`prompts = [f"a photo of a {c}" for c in classes]`** - build one prompt per class with a template; the wording is a real lever (prompt ensembling over several templates tends to help).
- **`proc(text=prompts, images=image, ...)`** - embed the image against all four class prompts in one pass.
- **`.logits_per_image.softmax(dim=1)[0]`** - probability over your class set (relative to these prompts, not absolute).
- **`probs.argmax()`** - the nearest class prompt is the prediction.

**In one line:** name your classes as prompts, embed the image against them, take the nearest.

**What you'll see:** Prints `predicted: dog  (0.96)` - CLIP classifies the sample image as 'dog' against the {dog, cat, car, pizza} label set.

## 6 - SigLIP 2: the 2026 successor

CLIP is the mental model, not the frontier. This cell swaps in SigLIP 2 (Google, sigmoid loss, multilingual) using the same `AutoModel` shape - a near one-line change - and highlights the key difference: a sigmoid instead of a softmax, so each image-caption score is an independent match probability that need not sum to 1.

> **Downloads pretrained weights** from Hugging Face on first run (no API key needed).

In [ ]:
# 2026: SigLIP 2 - the strong open successor to CLIP (sigmoid loss, multilingual).
from transformers import AutoModel, AutoProcessor
from PIL import Image
import torch

model = AutoModel.from_pretrained("google/siglip2-base-patch16-224")
proc  = AutoProcessor.from_pretrained("google/siglip2-base-patch16-224")

inputs = proc(text=["a photo of a dog", "a photo of a cat"],
              images=Image.open("dog.jpg"), return_tensors="pt", padding="max_length")
with torch.no_grad():
    logits = model(**inputs).logits_per_image

# SigLIP uses a SIGMOID: each score is an independent match probability, not a
# softmax over the row - so the two numbers need not sum to 1.
probs = torch.sigmoid(logits)[0]
print([round(p.item(), 2) for p in probs])
# Output: [0.19, 0.00]   (dog is the match; both are low - sigmoid is absolute, not a softmax that sums to 1)

**Explanation**

A swap-the-encoder demo showing that moving from CLIP to a stronger 2026 model is mostly a model-name change - but the scoring math differs. Where CLIP's softmax makes captions compete, SigLIP's sigmoid judges each pair on its own (matching the sigmoid loss idea from Step 4).

**How the code works - step by step**
- **`AutoModel.from_pretrained("google/siglip2-base-patch16-224")`** - load SigLIP 2 with the generic `Auto*` classes; same interface as CLIP.
- **`proc(..., padding="max_length")`** - SigLIP's processor expects max-length padding (a small API difference from CLIP).
- **`model(**inputs).logits_per_image`** - the raw per-pair match scores.
- **`torch.sigmoid(logits)`** - convert each score to an independent 0-1 match probability; the two numbers need not sum to 1, unlike CLIP's softmax.

**In one line:** same code shape as CLIP, but sigmoid (absolute per-pair scores) instead of softmax (row-normalized competition).

**What you'll see:** Prints a list like `[0.19, 0.00]` - dog is the match and cat is not; note both are low because sigmoid gives absolute per-pair scores rather than a softmax that sums to 1.

You built the whole vision-language stack from one idea: turn an image into a sequence of patch tokens (ViT), pool it to a single embedding, and train an image encoder and a text encoder so a picture and its caption land at the same point (CLIP). That shared space is the engine under everything you ran here - zero-shot classification, image search, and the contrastive objective that creates the space in the first place - and it is the same space that steered Stable Diffusion (6.2) and IP-Adapter (6.3). Next, Lesson 6.5 bolts a CLIP-style encoder onto a language model as its "eyes" to build multimodal LLMs; the 2026 encoders (SigLIP 2, DINOv3) swap the recipe but keep the idea unchanged.